# Phase 3 - Step 2: Vessel Segmentation (9 FSGNet)

## What Is FSG-Net?
FSG-Net (Full-Scale Guided Network) is a **2025 state-of-the-art CNN** published in BMC Medical Imaging (Seo et al., 2025).
It is specifically designed to solve the two main failures of standard U-Net for vessel segmentation:

1. **Loss of thin vessel detail** during downsampling in the encoder
2. **Weak boundary localization** in the decoder — blurry vessel edges

### Research Results (Official, 2025)
| Dataset | F1/Dice | AUC | Sensitivity | Accuracy |
|---|---|---|---|---|
| **DRIVE** | **84.068** | 98.235 | 84.207 | 97.042 |
| **STARE** | **85.100** | 98.967 | 86.608 | 97.746 |
| CHASE_DB1 | 81.019 | 99.378 | 85.995 | 97.515 |
| HRF | 81.567 | 98.744 | 83.616 | 97.106 |

FSG-Net outperforms TCDDU-Net (Swin Transformer hybrid, F1=82.65) despite being purely CNN — showing that smart architectural design can beat Transformers on this specific task.

### Two Core Innovations

**1. Feature Representation Network (FRN) — Full-Scale Module:**
Captures features at ALL scales simultaneously by merging feature maps from every encoder stage into one representation.
Unlike standard U-Net which only passes one scale at a time, FRN concatenates scales 1, 2, 3, and 4 together.
This gives the model both fine vessel detail (from early layers) and global anatomy (from deep layers) at the same time.

**2. Guided Convolution Block (GCB) — Attention-Guided Filter:**
After the FRN output, the GCB applies an attention-guided filter that works like **digital image sharpening (unsharp masking)**.
It generates an attention map that highlights thin vessel locations, then applies a guided residual filter to sharpen exactly those regions.
This is what makes FSG-Net exceptional at thin capillary recovery.

### Architecture Flow
```
Input → Encoder (4 stages) → Feature Representation Network (full-scale merge)
                                          ↓
                              Guided Convolution Block (attention-guided sharpening)
                                          ↓
                                   Final Mask Output
```

### Why This Is Relevant to Phase B
The FSG-Net GCB attention-guided filter is architecturally similar to your planned SCCSA skip connections.
Training FSG-Net in Phase A gives you a strong CNN baseline that directly informs which aspects of guided attention
improve thin vessel recovery — critical knowledge for your Phase B design.

In [ ]:
# ============================================================
# Cell 1: Setup & Installation
# ============================================================
!pip install -q segmentation-models-pytorch albumentations


In [ ]:
# ============================================================
# Cell 2: Imports
# ============================================================
import os
import json
import random
import zipfile
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
import albumentations as A
from albumentations.pytorch import ToTensorV2


In [ ]:
# ============================================================
# Cell 3: Reproducibility + Device
# ============================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

device = torch.device(
    'cuda' if torch.cuda.is_available() else
    'mps'  if torch.backends.mps.is_available() else
    'cpu'
)
print(f'Device : {device}')
if device.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')


In [ ]:
# ============================================================
# Cell 4: Load Preprocessed Dataset
# ============================================================
zip_path    = 'DR_dataset_processed.zip'
extract_dir = 'dataset'

with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall(extract_dir)

BASE_DIR   = extract_dir
TRAIN_CSV  = os.path.join(BASE_DIR,  'train_split.csv')
VAL_CSV    = os.path.join(BASE_DIR,  'val_split.csv')
TEST_CSV   = os.path.join(BASE_DIR,  'test_split.csv')


for split, path in [('Train', TRAIN_CSV), ('Val', VAL_CSV), ('Test', TEST_CSV)]:
    df = pd.read_csv(path)
    print(f'{split:<6}: {len(df):>4} samples | columns: {list(df.columns)}')


In [ ]:
# ============================================================
# Cell 5: Shared Modules (from src.dataset + src.metrics)
# ============================================================

# from src.dataset import RetinalDataset, get_train_transforms, get_val_transforms
# from src.metrics import dice_coefficient, iou_score, sensitivity, specificity,
#                         precision_score, compute_auc_roc, compute_auc_pr

"""
Retinal DR Detection - Shared Dataset + Metrics
=================================================
Same RetinalDataset and metric functions as other Phase A notebooks.
Augmentation is identical — fair comparison requires identical data pipeline.
"""

import os, cv2, numpy as np, pandas as pd, torch
from torch.utils.data import Dataset
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import roc_auc_score, average_precision_score


class RetinalDataset(Dataset):
    """
    PyTorch Dataset for retinal fundus images and segmentation masks.
    Reads image-mask pairs from a CSV split file.
    Identical to all other Phase A notebooks for fair model comparison.
    """
    def __init__(self, csv_file, base_dir, transform=None, mask_col='vessel_path'):
        self.df        = pd.read_csv(csv_file)
        self.base_dir  = base_dir
        self.transform = transform
        self.mask_col  = mask_col

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row       = self.df.iloc[idx]
        img_path  = os.path.join(self.base_dir, row['img_path'])
        mask_path = os.path.join(self.base_dir, row[self.mask_col])

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        mask = (mask > 127).astype(np.float32)

        if self.transform is not None:
            aug   = self.transform(image=image, mask=mask)
            image = aug['image']
            mask  = aug['mask']

        if isinstance(mask, torch.Tensor):
            mask = mask.unsqueeze(0)
        else:
            mask = torch.from_numpy(mask).unsqueeze(0)

        return image, mask


def get_train_transforms(img_size=512):
    """
    Training augmentation — identical across all Phase A models.
    Geometric + color transforms with additional_targets for mask sync.
    FSG-Net specifically benefits from ElasticTransform because the
    attention-guided filter learns to sharpen curved vessel structures.
    """
    return A.Compose([
        A.Resize(img_size, img_size),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.ShiftScaleRotate(
            shift_limit=0.0625, scale_limit=0.1, rotate_limit=45,
            border_mode=cv2.BORDER_CONSTANT, p=0.5
        ),
        A.GridDistortion(num_steps=5, distort_limit=0.3, p=0.3),
        A.ElasticTransform(alpha=120, sigma=120*0.05, p=0.2),
        # ElasticTransform is especially valuable for FSG-Net:
        # the GCB attention filter learns to sharpen thin deformed vessels
        A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.4),
        A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8), p=0.3),
        A.GaussianBlur(blur_limit=(3, 5), p=0.2),
        A.GaussNoise(var_limit=(5, 25), p=0.2),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ], additional_targets={'mask': 'mask'})


def get_val_transforms(img_size=512):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ], additional_targets={'mask': 'mask'})


# ── Metrics ─────────────────────────────────────────────────────────────────

def dice_coefficient(pred, target, smooth=1e-6):
    """Dice = 2*|P∩T| / (|P|+|T|). Main segmentation metric."""
    pf = pred.contiguous().view(-1).float()
    tf = target.contiguous().view(-1).float()
    return (2.*(pf*tf).sum() + smooth) / (pf.sum() + tf.sum() + smooth)

def iou_score(pred, target, smooth=1e-6):
    """IoU (Jaccard) = |P∩T| / |P∪T|. Stricter than Dice."""
    pf = pred.contiguous().view(-1).float()
    tf = target.contiguous().view(-1).float()
    inter = (pf*tf).sum()
    return (inter + smooth) / (pf.sum() + tf.sum() - inter + smooth)

def sensitivity(pred, target, smooth=1e-6):
    """Sensitivity (Recall) = TP/(TP+FN)."""
    pf = pred.contiguous().view(-1).float()
    tf = target.contiguous().view(-1).float()
    tp = (pf*tf).sum(); fn = (tf*(1-pf)).sum()
    return (tp + smooth) / (tp + fn + smooth)

def specificity(pred, target, smooth=1e-6):
    """Specificity (TNR) = TN/(TN+FP)."""
    pf = pred.contiguous().view(-1).float()
    tf = target.contiguous().view(-1).float()
    tn = ((1-pf)*(1-tf)).sum(); fp = (pf*(1-tf)).sum()
    return (tn + smooth) / (tn + fp + smooth)

def precision_score(pred, target, smooth=1e-6):
    """Precision (PPV) = TP/(TP+FP)."""
    pf = pred.contiguous().view(-1).float()
    tf = target.contiguous().view(-1).float()
    tp = (pf*tf).sum(); fp = (pf*(1-tf)).sum()
    return (tp + smooth) / (tp + fp + smooth)

def compute_auc_roc(probs, labels):
    try: return roc_auc_score(labels, probs)
    except: return 0.0

def compute_auc_pr(probs, labels):
    try: return average_precision_score(labels, probs)
    except: return 0.0


print('Shared modules ready (RetinalDataset, transforms, metrics)')


In [ ]:
# ============================================================
# Cell 6: Hyperparameter Configuration
# ============================================================

CFG = dict(
    # Paths
    base_dir    = 'dataset',
    train_csv   = 'dataset/train_split.csv',
    val_csv     = 'dataset/val_split.csv',
    test_csv    = 'dataset/test_split.csv',
    results_dir = 'FSG_Net_results',

    # Model
    # FSG-Net uses a 4-stage encoder (not 5 like standard U-Net).
    # The paper found wider receptive field is LESS critical for vessel seg
    # than fine detail preservation — so fewer downsampling stages.
    in_channels      = 3,
    out_channels     = 1,
    img_size         = 512,
    encoder_channels = [32, 64, 128, 256],   # 4-stage encoder channel sizes
    base_channels    = 32,                   # starting channel count
    gcb_channels     = 64,                   # guided conv block hidden dim

    # Training — two phases (same strategy as Attention U-Net)
    freeze_epochs  = 5,
    total_epochs   = 80,
    batch_size     = 4,
    num_workers    = 4,

    # Learning rates
    lr_encoder    = 1e-5,
    lr_decoder    = 3e-4,
    weight_decay  = 1e-4,
    grad_clip     = 1.0,
    use_amp       = True,
    warmup_epochs = 3,

    # Loss
    dice_weight     = 0.6,
    bce_weight      = 0.4,
    label_smoothing = 0.05,

    # FSG-Net specific: deep supervision
    # The paper uses auxiliary losses on intermediate decoder outputs
    # to force the network to produce good predictions at multiple scales.
    # This is what y2 and y3 refer to in the original paper.
    use_deep_supervision = True,
    ds_weight            = 0.4,    # weight for auxiliary losses

    # Early stopping
    patience  = 12,
    threshold = 0.5,
    seed      = 42,
)

for sub in ['checkpoints', 'plots', 'predictions', 'metrics']:
    os.makedirs(os.path.join(CFG['results_dir'], sub), exist_ok=True)

print('Configuration ready')
print(f'Results: {CFG["results_dir"]}/')
print(f'Model  : FSG-Net (4-stage encoder + FRN + GCB)')
print(f'Paper  : F1=84.068 on DRIVE  |  F1=85.100 on STARE  (Seo et al., 2025)')


In [ ]:
# ============================================================
# Cell 7: Build DataLoaders
# ============================================================

train_dataset = RetinalDataset(CFG['train_csv'], CFG['base_dir'],
                               transform=get_train_transforms(CFG['img_size']))
val_dataset   = RetinalDataset(CFG['val_csv'],   CFG['base_dir'],
                               transform=get_val_transforms(CFG['img_size']))
test_dataset  = RetinalDataset(CFG['test_csv'],  CFG['base_dir'],
                               transform=get_val_transforms(CFG['img_size']))

train_loader = DataLoader(train_dataset, batch_size=CFG['batch_size'], shuffle=True,
    num_workers=CFG['num_workers'], pin_memory=True, persistent_workers=True,
    drop_last=True, worker_init_fn=lambda w: np.random.seed(CFG['seed']+w))
val_loader  = DataLoader(val_dataset,  batch_size=CFG['batch_size'], shuffle=False,
    num_workers=CFG['num_workers'], pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=CFG['batch_size'], shuffle=False,
    num_workers=CFG['num_workers'], pin_memory=True)

print(f'Train: {len(train_dataset)} samples | {len(train_loader)} batches')
print(f'Val  : {len(val_dataset)} samples   | {len(val_loader)} batches')
print(f'Test : {len(test_dataset)} samples  | {len(test_loader)} batches')

imgs, masks = next(iter(train_loader))
print(f'\nBatch: images={tuple(imgs.shape)}  masks={tuple(masks.shape)}')
print(f'Vessel ratio: {masks.mean():.4f} ({masks.mean()*100:.2f}% of pixels)')


## FSG-Net Architecture — Deep Dive

The architecture has three main parts:

### Part 1 — Modernized Convolution Block (MCB)
Used inside the encoder instead of standard conv-BN-ReLU. Each MCB contains:
- **Depthwise separable convolutions** — factorizes conv into depthwise + pointwise, reducing parameters by ~8x
- **Batch Normalization + GELU activation** — GELU outperforms ReLU for fine vessel boundaries
- **Residual connection** — prevents gradient vanishing in deep encoder

### Part 2 — Feature Representation Network (FRN) — Full-Scale Module
This is the key innovation. At every decoder stage it collects feature maps from **all encoder scales** and merges them:
```
Stage 1 features (high-res, fine detail) ──────────────────────┐
Stage 2 features (medium-res)            → upsample to same → concatenate → 1x1 conv
Stage 3 features (low-res)               → upsample to same →
Stage 4 features (bottleneck, semantic)  → upsample to same →
```
Result: ONE feature map that simultaneously has fine vessel detail AND global anatomy context.

### Part 3 — Guided Convolution Block (GCB) — Attention-Guided Filter
Applied after the FRN. Works like digital image unsharp masking:
```
Full-scale features → Attention branch → attention map (where are thin vessels?)
                   ↓
              Guided residual module: sharpen exactly those locations
                   ↓
              Deep supervision: auxiliary loss at this scale
```
The attention map is learned — it specializes in highlighting low-contrast thin capillaries.
The guided filter then SHARPENS those regions while leaving background unchanged.

### Deep Supervision
The paper adds **auxiliary segmentation heads** at intermediate decoder stages (y2, y3).
This forces the network to produce correct predictions at multiple scales,
not just the final output. Benefits: faster convergence, stronger gradient flow to early layers.

In [ ]:
# ============================================================
# Cell 9: FSG-Net — Fixed PyTorch Implementation
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F


# ── Block 1: Depthwise Separable Conv ───────────────────────
class DepthwiseSeparableConv(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=3, padding=1):
        super().__init__()
        self.depthwise = nn.Conv2d(
            in_ch, in_ch, kernel_size,
            padding=padding, groups=in_ch, bias=False
        )
        self.pointwise = nn.Conv2d(in_ch, out_ch, 1, bias=False)

    def forward(self, x):
        return self.pointwise(self.depthwise(x))


# ── Block 2: Modernized Convolution Block (MCB) ─────────────
class MCB(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv1 = DepthwiseSeparableConv(in_ch, out_ch)
        self.bn1   = nn.BatchNorm2d(out_ch)

        self.conv2 = DepthwiseSeparableConv(out_ch, out_ch)
        self.bn2   = nn.BatchNorm2d(out_ch)

        self.act = nn.GELU()

        self.shortcut = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch)
        ) if in_ch != out_ch else nn.Identity()

    def forward(self, x):
        residual = self.shortcut(x)
        out = self.act(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return self.act(out + residual)


# ── Block 3: Full-Scale Module ──────────────────────────────
class FullScaleModule(nn.Module):
    def __init__(self, encoder_channels, out_ch):
        super().__init__()
        total_in = sum(encoder_channels)
        self.fuse = nn.Sequential(
            nn.Conv2d(total_in, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.GELU(),
        )

    def forward(self, enc_features, target_size):
        upsampled = [
            F.interpolate(f, size=target_size, mode='bilinear', align_corners=False)
            for f in enc_features
        ]
        fused = torch.cat(upsampled, dim=1)
        return self.fuse(fused)


# ── Block 4: Guided Residual Module (FIXED) ─────────────────
class GuidedResidualModule(nn.Module):
    """
    x_ch   : channels of decoder feature x
    ctx_ch : channels of full-scale feature full_scale_feat
    """
    def __init__(self, x_ch, ctx_ch, hidden_ch):
        super().__init__()

        # Attention is generated FROM full-scale context
        # and converted TO decoder feature channel dimension
        self.attn = nn.Sequential(
            nn.Conv2d(ctx_ch, hidden_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(hidden_ch),
            nn.GELU(),
            nn.Conv2d(hidden_ch, x_ch, 1, bias=False),
            nn.Sigmoid(),
        )

        # Guided refinement operates on decoder feature channels
        self.guided_conv = nn.Sequential(
            nn.Conv2d(x_ch, hidden_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(hidden_ch),
            nn.GELU(),
            nn.Conv2d(hidden_ch, x_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(x_ch),
        )

        self.act = nn.GELU()

    def forward(self, x, full_scale_feat):
        attn_map = self.attn(full_scale_feat)   # (B, x_ch, H, W)
        guided   = self.guided_conv(x * attn_map)
        return self.act(x + guided)


# ── Block 5: Guided Convolution Block (FIXED) ───────────────
class GuidedConvBlock(nn.Module):
    def __init__(self, in_ch, ctx_ch, hidden_ch, out_ch, n_grm=2):
        super().__init__()
        self.grms = nn.ModuleList([
            GuidedResidualModule(in_ch, ctx_ch, hidden_ch)
            for _ in range(n_grm)
        ])
        self.ds_head = nn.Conv2d(in_ch, out_ch, 1)

    def forward(self, x, full_scale_feat):
        for grm in self.grms:
            x = grm(x, full_scale_feat)
        aux_pred = self.ds_head(x)
        return x, aux_pred


# ── Main FSG-Net ────────────────────────────────────────────
class FSGNet(nn.Module):
    def __init__(self, in_ch=3, out_ch=1, enc_channels=None, gcb_hidden=64):
        super().__init__()

        if enc_channels is None:
            enc_channels = [32, 64, 128, 256]

        # Encoder
        self.enc1 = MCB(in_ch, enc_channels[0])           # 512x512
        self.enc2 = MCB(enc_channels[0], enc_channels[1]) # 256x256
        self.enc3 = MCB(enc_channels[1], enc_channels[2]) # 128x128
        self.enc4 = MCB(enc_channels[2], enc_channels[3]) # 64x64
        self.pool = nn.MaxPool2d(2, 2)

        # Full-scale modules
        self.fsm_out = 128
        self.fsm3 = FullScaleModule(enc_channels, self.fsm_out)
        self.fsm2 = FullScaleModule(enc_channels, self.fsm_out)
        self.fsm1 = FullScaleModule(enc_channels, self.fsm_out)

        # Decoder
        self.up4  = nn.ConvTranspose2d(enc_channels[3], enc_channels[2], 2, stride=2)
        self.dec3 = MCB(enc_channels[2] * 2, enc_channels[2])

        self.up3  = nn.ConvTranspose2d(enc_channels[2], enc_channels[1], 2, stride=2)
        self.dec2 = MCB(enc_channels[1] * 2, enc_channels[1])

        self.up2  = nn.ConvTranspose2d(enc_channels[1], enc_channels[0], 2, stride=2)
        self.dec1 = MCB(enc_channels[0] * 2, enc_channels[0])

        # Guided Convolution Blocks (FIXED: ctx_ch=self.fsm_out)
        self.gcb3 = GuidedConvBlock(
            in_ch=enc_channels[2],
            ctx_ch=self.fsm_out,
            hidden_ch=gcb_hidden,
            out_ch=out_ch,
            n_grm=2
        )
        self.gcb2 = GuidedConvBlock(
            in_ch=enc_channels[1],
            ctx_ch=self.fsm_out,
            hidden_ch=gcb_hidden,
            out_ch=out_ch,
            n_grm=2
        )
        self.gcb1 = GuidedConvBlock(
            in_ch=enc_channels[0],
            ctx_ch=self.fsm_out,
            hidden_ch=gcb_hidden,
            out_ch=out_ch,
            n_grm=2
        )

        self.final_head = nn.Conv2d(enc_channels[0], out_ch, 1)

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)               # (B, 32, 512, 512)
        e2 = self.enc2(self.pool(e1))   # (B, 64, 256, 256)
        e3 = self.enc3(self.pool(e2))   # (B, 128, 128, 128)
        e4 = self.enc4(self.pool(e3))   # (B, 256, 64, 64)

        enc_feats = [e1, e2, e3, e4]

        # Decoder stage 3
        d3 = self.up4(e4)                       # (B, 128, 128, 128)
        d3 = torch.cat([d3, e3], dim=1)         # (B, 256, 128, 128)
        d3 = self.dec3(d3)                      # (B, 128, 128, 128)
        fs3 = self.fsm3(enc_feats, d3.shape[2:])# (B, 128, 128, 128)
        d3, aux3 = self.gcb3(d3, fs3)           # aux3: (B, 1, 128, 128)

        # Decoder stage 2
        d2 = self.up3(d3)                       # (B, 64, 256, 256)
        d2 = torch.cat([d2, e2], dim=1)         # (B, 128, 256, 256)
        d2 = self.dec2(d2)                      # (B, 64, 256, 256)
        fs2 = self.fsm2(enc_feats, d2.shape[2:])# (B, 128, 256, 256)
        d2, aux2 = self.gcb2(d2, fs2)           # aux2: (B, 1, 256, 256)

        # Decoder stage 1
        d1 = self.up2(d2)                       # (B, 32, 512, 512)
        d1 = torch.cat([d1, e1], dim=1)         # (B, 64, 512, 512)
        d1 = self.dec1(d1)                      # (B, 32, 512, 512)
        fs1 = self.fsm1(enc_feats, d1.shape[2:])# (B, 128, 512, 512)
        d1, aux1 = self.gcb1(d1, fs1)           # aux1: (B, 1, 512, 512)

        # Final output
        out = self.final_head(d1)               # (B, 1, 512, 512)

        return out, aux1, aux2, aux3


# ============================================================
# Build model and run sanity check
# ============================================================
model = FSGNet(
    in_ch=CFG['in_channels'],
    out_ch=CFG['out_channels'],
    enc_channels=CFG['encoder_channels'],
    gcb_hidden=CFG['gcb_channels'],
).to(device)

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("FSG-Net architecture built successfully")
print(f"Total params    : {total:,}")
print(f"Trainable params: {trainable:,}")

with torch.no_grad():
    dummy = torch.randn(2, 3, 512, 512).to(device)
    out, a1, a2, a3 = model(dummy)

    print("\nForward pass check:")
    print(f"  Main output : {tuple(out.shape)}   (should be [2, 1, 512, 512])")
    print(f"  Aux pred 1  : {tuple(a1.shape)}   (full-res auxiliary)")
    print(f"  Aux pred 2  : {tuple(a2.shape)}   (1/2-res auxiliary)")
    print(f"  Aux pred 3  : {tuple(a3.shape)}   (1/4-res auxiliary)")

In [ ]:
# ============================================================
# Cell 10: FSG-Net Loss Function with Deep Supervision
# ============================================================
# FSG-Net adds deep supervision: auxiliary losses at intermediate scales.
# Total loss = main_loss + ds_weight * (aux1_loss + aux2_loss + aux3_loss)


class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, preds, targets):
        preds = torch.sigmoid(preds)
        pf    = preds.view(preds.size(0), -1)
        tf    = targets.view(targets.size(0), -1)
        inter = (pf * tf).sum(dim=1)
        return 1. - ((2.*inter + self.smooth) / (pf.sum(dim=1) + tf.sum(dim=1) + self.smooth)).mean()


class FSGNetLoss(nn.Module):
    """
    FSG-Net combined loss with deep supervision.

    Main loss:     Dice + BCE on final output (512x512)
    Aux losses:    Dice + BCE on 3 auxiliary outputs at different scales

    Why deep supervision helps:
    - Forces intermediate decoder stages to produce good vessel predictions
    - Provides gradient directly to early layers (no vanishing gradient)
    - Auxiliary targets are downsampled from the original mask
    - At inference: only the final output is used, aux heads discarded
    """
    def __init__(self, dice_w=0.6, bce_w=0.4, ds_w=0.4, label_smooth=0.05):
        super().__init__()
        self.dice_w = dice_w
        self.bce_w  = bce_w
        self.ds_w   = ds_w
        self.ls     = label_smooth
        self.dice   = DiceLoss()
        self.bce    = nn.BCEWithLogitsLoss()

    def _single_loss(self, pred, target):
        """Compute Dice + BCE for one prediction-target pair."""
        smooth_t = target * (1 - self.ls) + 0.5 * self.ls
        return self.dice_w * self.dice(pred, target) + self.bce_w * self.bce(pred, smooth_t)

    def forward(self, out, aux1, aux2, aux3, target):
        """
        out:   main output    (B, 1, H, W)
        aux1:  full-res aux   (B, 1, H, W)
        aux2:  1/2 res aux    (B, 1, H/2, W/2)
        aux3:  1/4 res aux    (B, 1, H/4, W/4)
        target: ground truth  (B, 1, H, W)
        """
        # Main loss
        main_loss = self._single_loss(out, target)

        # Auxiliary losses — downsample target to match aux output sizes
        t2 = F.interpolate(target, size=aux2.shape[2:], mode='bilinear', align_corners=False)
        t3 = F.interpolate(target, size=aux3.shape[2:], mode='bilinear', align_corners=False)

        aux1_loss = self._single_loss(aux1, target)  # same size as main
        aux2_loss = self._single_loss(aux2, t2)
        aux3_loss = self._single_loss(aux3, t3)

        total = main_loss + self.ds_w * (aux1_loss + aux2_loss + aux3_loss) / 3.0
        return total, main_loss, aux1_loss


criterion = FSGNetLoss(
    dice_w=CFG['dice_weight'], bce_w=CFG['bce_weight'],
    ds_w=CFG['ds_weight'], label_smooth=CFG['label_smoothing']
)
print('FSG-Net loss with deep supervision ready')
print(f'  Main loss weight : 1.0  (Dice×{CFG["dice_weight"]} + BCE×{CFG["bce_weight"]})')
print(f'  Aux loss weight  : {CFG["ds_weight"]} × (aux1 + aux2 + aux3) / 3')


In [ ]:
# ============================================================
# Cell 11: Optimizer + Scheduler
# ============================================================
# FSG-Net is trained from scratch (no ImageNet pretrained encoder).
# We use a single LR initially, then apply warmup+cosine schedule.
# Two-phase: freeze nothing in phase 1 (all random weights, need full training)
# but use lower LR for encoder after warmup.

def build_optimizer(model, phase=1):
    if phase == 1:
        # Phase 1: train ALL parameters with uniform LR
        # FSG-Net has no pretrained weights — all parts need training
        return optim.AdamW(model.parameters(),
                           lr=CFG['lr_decoder'],
                           weight_decay=CFG['weight_decay'])
    else:
        # Phase 2: differential LR — encoder slightly lower
        encoder_params = list(model.enc1.parameters()) + list(model.enc2.parameters()) + \
                         list(model.enc3.parameters()) + list(model.enc4.parameters())
        other_params   = [p for n, p in model.named_parameters()
                          if not any(n.startswith(e) for e in ['enc1','enc2','enc3','enc4'])]
        return optim.AdamW([
            {'params': encoder_params, 'lr': CFG['lr_encoder']},
            {'params': other_params,   'lr': CFG['lr_decoder']},
        ], weight_decay=CFG['weight_decay'])


def build_scheduler(optimizer, total_epochs, warmup_epochs):
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return (epoch + 1) / warmup_epochs
        progress = (epoch - warmup_epochs) / max(1, total_epochs - warmup_epochs)
        return 0.5 * (1 + np.cos(np.pi * progress))
    return optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


def save_checkpoint(model, optimizer, scheduler, epoch, metrics, filename):
    path = os.path.join(CFG['results_dir'], 'checkpoints', filename)
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'metrics': metrics,
        'config': CFG,
    }, path)
    print(f'  Checkpoint: {filename}  (Dice: {metrics["dice"]:.4f})')


def load_best_model(model, filename='best_model.pt'):
    path = os.path.join(CFG['results_dir'], 'checkpoints', filename)
    # weights_only=False required for PyTorch 2.6+ with numpy scalars in checkpoint
    ckpt = torch.load(path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    print(f'Loaded best model from epoch {ckpt["epoch"]}  (Val Dice: {ckpt["metrics"]["dice"]:.4f})')
    return ckpt['metrics']


print('Optimizer and checkpointing functions ready')


In [ ]:
# ============================================================
# Cell 12: Training + Validation Loops
# ============================================================

def train_one_epoch(model, loader, optimizer, criterion, scaler, epoch):
    model.train()
    total_loss = total_main = 0.0
    all_dice   = []
    n = len(loader)

    for step, (images, masks) in enumerate(loader):
        images = images.to(device, non_blocking=True)
        masks  = masks.to(device,  non_blocking=True)
        optimizer.zero_grad()

        with  autocast(enabled=(scaler is not None)):
            # FSG-Net returns 4 outputs: main + 3 auxiliary
            out, aux1, aux2, aux3 = model(images)
            loss, main_loss, _ = criterion(out, aux1, aux2, aux3, masks)

        if scaler is not None:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), CFG['grad_clip'])
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), CFG['grad_clip'])
            optimizer.step()

        total_loss += loss.item()
        total_main += main_loss.item()

        with torch.no_grad():
            pred_bin = (torch.sigmoid(out.detach()) > CFG['threshold']).float()
            all_dice.append(dice_coefficient(pred_bin, masks).item())

        if (step + 1) % 20 == 0:
            print(f'  Ep{epoch} [{step+1}/{n}] total={loss.item():.4f} main={main_loss.item():.4f} dice={all_dice[-1]:.4f}')

    return {'loss': total_loss/n, 'main_loss': total_main/n, 'dice': np.mean(all_dice)}


@torch.no_grad()
def validate(model, loader):
    model.eval()
    total_loss = 0.0
    all_dice = []; all_iou = []; all_sens = []; all_spec = []; all_prec = []
    all_probs = []; all_labels = []
    n = len(loader)

    for images, masks in loader:
        images = images.to(device, non_blocking=True)
        masks  = masks.to(device,  non_blocking=True)

        # At inference: only main output matters — auxiliary heads discarded
        out, aux1, aux2, aux3 = model(images)
        loss, _, _ = criterion(out, aux1, aux2, aux3, masks)
        total_loss += loss.item()

        probs    = torch.sigmoid(out)
        pred_bin = (probs > CFG['threshold']).float()

        all_dice.append(dice_coefficient(pred_bin, masks).item())
        all_iou.append(iou_score(pred_bin, masks).item())
        all_sens.append(sensitivity(pred_bin, masks).item())
        all_spec.append(specificity(pred_bin, masks).item())
        all_prec.append(precision_score(pred_bin, masks).item())
        all_probs.append(probs.cpu().numpy().reshape(-1))
        all_labels.append(masks.cpu().numpy().reshape(-1))

    probs_flat  = np.concatenate(all_probs)
    labels_flat = np.concatenate(all_labels).astype(int)

    return {
        'loss'       : total_loss / n,
        'dice'       : round(np.mean(all_dice), 4),
        'iou'        : round(np.mean(all_iou), 4),
        'sensitivity': round(np.mean(all_sens), 4),
        'specificity': round(np.mean(all_spec), 4),
        'precision'  : round(np.mean(all_prec), 4),
        'auc_roc'    : round(compute_auc_roc(probs_flat, labels_flat), 4),
        'auc_pr'     : round(compute_auc_pr(probs_flat, labels_flat), 4),
        '_probs'     : probs_flat,
        '_labels'    : labels_flat,
    }


print('Training and validation loops ready')


In [ ]:
import gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# ============================================================
# Cell 13: Main Training — Two-Phase Strategy
# ============================================================

set_seed(CFG['seed'])
scaler   = GradScaler() if CFG['use_amp'] and device.type == 'cuda' else None
history  = []
best_dice = 0.0
patience_counter = 0

print('=' * 60)
print('PHASE 1: Full training — all layers (FSG-Net has no pretrained weights)')
print('=' * 60)

optimizer = build_optimizer(model, phase=1)
scheduler = build_scheduler(optimizer, CFG['freeze_epochs'], warmup_epochs=1)

for epoch in range(1, CFG['freeze_epochs'] + 1):
    tr = train_one_epoch(model, train_loader, optimizer, criterion, scaler, epoch)
    vl = validate(model, val_loader)
    scheduler.step()
    lr_now = scheduler.get_last_lr()[0]

    row = {'phase': 1, 'epoch': epoch, 'lr': lr_now,
           **{f'train_{k}': v for k, v in tr.items()},
           **{f'val_{k}': v for k, v in vl.items() if not k.startswith('_')}}
    history.append(row)
    print(f'  Ep {epoch:02d}/{CFG["freeze_epochs"]} | '
          f'Train Dice: {tr["dice"]:.4f} | Val Dice: {vl["dice"]:.4f} | '
          f'IoU: {vl["iou"]:.4f} | Sens: {vl["sensitivity"]:.4f} | LR: {lr_now:.2e}')
    if vl['dice'] > best_dice:
        best_dice = vl['dice']
        save_checkpoint(model, optimizer, scheduler, epoch, vl, 'best_model.pt')

print(f'\nPhase 1 done. Best Val Dice: {best_dice:.4f}')
print('\n' + '='*60)
print('PHASE 2: Fine-tuning with differential LR')
print('='*60)

optimizer = build_optimizer(model, phase=2)
remaining = CFG['total_epochs'] - CFG['freeze_epochs']
scheduler = build_scheduler(optimizer, remaining, CFG['warmup_epochs'])
patience_counter = 0

for epoch in range(CFG['freeze_epochs'] + 1, CFG['total_epochs'] + 1):
    tr = train_one_epoch(model, train_loader, optimizer, criterion, scaler, epoch)
    vl = validate(model, val_loader)
    scheduler.step()
    lr_enc = optimizer.param_groups[0]['lr']
    lr_dec = optimizer.param_groups[1]['lr']

    row = {'phase': 2, 'epoch': epoch, 'lr_encoder': lr_enc, 'lr_decoder': lr_dec,
           **{f'train_{k}': v for k, v in tr.items()},
           **{f'val_{k}': v for k, v in vl.items() if not k.startswith('_')}}
    history.append(row)
    print(f'  Ep {epoch:02d}/{CFG["total_epochs"]} | '
          f'Train Dice: {tr["dice"]:.4f} | Val Dice: {vl["dice"]:.4f} | '
          f'Sens: {vl["sensitivity"]:.4f} | LR enc={lr_enc:.1e} dec={lr_dec:.1e}')

    if vl['dice'] > best_dice:
        best_dice = vl['dice']
        patience_counter = 0
        save_checkpoint(model, optimizer, scheduler, epoch, vl, 'best_model.pt')
    else:
        patience_counter += 1
        if patience_counter >= CFG['patience']:
            print(f'Early stopping at epoch {epoch}. Best Val Dice: {best_dice:.4f}')
            break

    if epoch % 10 == 0:
        save_checkpoint(model, optimizer, scheduler, epoch, vl, f'epoch_{epoch:03d}.pt')

history_path = os.path.join(CFG['results_dir'], 'metrics', 'training_history.json')
with open(history_path, 'w') as f:
    json.dump(history, f, indent=2)
print(f'Training done. Best Val Dice: {best_dice:.4f}')


In [ ]:
# ============================================================
# Cell 14: Plot 1 — Training History
# ============================================================

history_path = os.path.join(CFG['results_dir'], 'metrics', 'training_history.json')
with open(history_path) as f:
    hist = json.load(f)

epochs     = [h['epoch']                  for h in hist]
train_loss = [h.get('train_loss', 0)      for h in hist]
val_loss   = [h.get('val_loss', 0)        for h in hist]
train_dice = [h.get('train_dice', 0)      for h in hist]
val_dice   = [h.get('val_dice', 0)        for h in hist]
val_iou    = [h.get('val_iou', 0)         for h in hist]
val_sens   = [h.get('val_sensitivity', 0) for h in hist]
val_spec   = [h.get('val_specificity', 0) for h in hist]
val_auc    = [h.get('val_auc_roc', 0)     for h in hist]
phase_boundary = next((h['epoch'] for h in hist if h.get('phase') == 2), None)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('FSG-Net — Training History (Full-Scale Guided Network)', fontsize=15, fontweight='bold')

ax = axes[0, 0]
ax.plot(epochs, train_loss, label='Train loss', color='#2196F3', lw=2)
ax.plot(epochs, val_loss,   label='Val loss',   color='#FF5722', lw=2)
if phase_boundary:
    ax.axvline(x=phase_boundary, color='gray', ls='--', lw=1.5, label='Phase 2 start')
ax.set_title('Total Loss (Main + Deep Supervision)', fontsize=12)
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[0, 1]
ax.plot(epochs, train_dice, label='Train Dice', color='#2196F3', lw=2)
ax.plot(epochs, val_dice,   label='Val Dice',   color='#FF5722', lw=2)
ax.axhline(y=0.84, color='#4CAF50', ls='--', lw=1.5, label='Paper target (DRIVE 0.84068)')
ax.axhline(y=0.82, color='gray',    ls=':',  lw=1.2, label='Phase A target (0.82)')
if phase_boundary:
    ax.axvline(x=phase_boundary, color='gray', ls='--', lw=1.5)
ax.set_title('Dice Score', fontsize=12)
ax.set_xlabel('Epoch'); ax.set_ylabel('Dice')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3); ax.set_ylim([0, 1])

ax = axes[1, 0]
ax.plot(epochs, val_dice, label='Dice',        color='#2196F3', lw=2)
ax.plot(epochs, val_iou,  label='IoU',         color='#9C27B0', lw=2)
ax.plot(epochs, val_sens, label='Sensitivity', color='#FF9800', lw=2)
ax.plot(epochs, val_spec, label='Specificity', color='#4CAF50', lw=2)
ax.plot(epochs, val_auc,  label='AUC-ROC',     color='#E91E63', lw=2)
ax.set_title('All Validation Metrics', fontsize=12)
ax.set_xlabel('Epoch'); ax.set_ylabel('Score')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3); ax.set_ylim([0, 1])

ax = axes[1, 1]
main_loss_h = [h.get('train_main_loss', 0) for h in hist]
ax.plot(epochs, train_loss,  label='Total loss (main+DS)', color='#FF5722', lw=2)
ax.plot(epochs, main_loss_h, label='Main loss only',       color='#2196F3', lw=2)
ax.set_title('Total vs Main Loss (Deep Supervision Effect)', fontsize=12)
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
out = os.path.join(CFG['results_dir'], 'plots', 'training_history.png')
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Training curves saved → {out}')


In [ ]:
# ============================================================
# Cell 15: Plot 2 — ROC and PR Curves
# ============================================================

best_ckpt_metrics = load_best_model(model)

print('Collecting test set predictions...')
test_results = validate(model, test_loader)

probs  = test_results['_probs']
labels = test_results['_labels']
idx    = np.random.choice(len(probs), min(500_000, len(probs)), replace=False)
probs_s  = probs[idx]
labels_s = labels[idx].astype(int)

fpr, tpr, _  = roc_curve(labels_s, probs_s)
prec, rec, _ = precision_recall_curve(labels_s, probs_s)
roc_auc      = test_results['auc_roc']
auc_pr       = test_results['auc_pr']
vessel_ratio = labels_s.mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('FSG-Net — ROC and PR Curves (Test Set)', fontsize=14, fontweight='bold')

ax = axes[0]
ax.plot(fpr, tpr, color='#2196F3', lw=2.5, label=f'AUC-ROC = {roc_auc:.4f}')
ax.plot([0, 1], [0, 1], color='gray', ls='--', lw=1)
ax.fill_between(fpr, tpr, alpha=0.08, color='#2196F3')
ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])
ax.set_xlabel('False Positive Rate  (1 - Specificity)', fontsize=11)
ax.set_ylabel('True Positive Rate  (Sensitivity)', fontsize=11)
ax.set_title('ROC Curve', fontsize=13)
ax.legend(loc='lower right', fontsize=11)
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(rec, prec, color='#FF5722', lw=2.5, label=f'AUC-PR = {auc_pr:.4f}')
ax.axhline(y=vessel_ratio, color='gray', ls='--', lw=1,
           label=f'Random baseline ({vessel_ratio:.3f})')
ax.fill_between(rec, prec, alpha=0.08, color='#FF5722')
ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])
ax.set_xlabel('Recall  (Sensitivity)', fontsize=11)
ax.set_ylabel('Precision', fontsize=11)
ax.set_title('Precision-Recall Curve', fontsize=13)
ax.legend(loc='upper right', fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
out = os.path.join(CFG['results_dir'], 'plots', 'roc_pr_curves.png')
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'ROC/PR curves saved → {out}')
print(f'AUC-ROC: {roc_auc:.4f}  |  AUC-PR: {auc_pr:.4f}')


In [ ]:
# ============================================================
# Cell 16: Plot 3 — Prediction Visualization
# ============================================================

MEAN = np.array([0.485, 0.456, 0.406])
STD  = np.array([0.229, 0.224, 0.225])

@torch.no_grad()
def visualize_predictions(model, loader, n_samples=8, save_dir=None):
    model.eval()
    save_dir = save_dir or os.path.join(CFG['results_dir'], 'predictions')
    os.makedirs(save_dir, exist_ok=True)
    count = 0

    for images, masks in loader:
        if count >= n_samples:
            break
        images_gpu = images.to(device, non_blocking=True)
        # FSG-Net: take only main output, discard aux
        out, _, _, _ = model(images_gpu)
        probs_np = torch.sigmoid(out).cpu().numpy()
        images_np = images.numpy()
        masks_np  = masks.numpy()

        for i in range(min(images.size(0), n_samples - count)):
            img  = np.clip(images_np[i].transpose(1,2,0) * STD + MEAN, 0, 1)
            gt   = masks_np[i, 0]
            prob = probs_np[i, 0]
            pred = (prob > CFG['threshold']).astype(float)

            overlay = img.copy()
            overlay[pred > 0.5] = [1.0, 0.15, 0.15]

            err = np.zeros((*gt.shape, 3))
            err[..., 1] = pred * gt
            err[..., 0] = pred * (1 - gt)
            err[..., 2] = (1 - pred) * gt

            inter    = (pred * gt).sum()
            img_dice = (2*inter + 1) / (pred.sum() + gt.sum() + 1)
            img_sens = (pred * gt).sum() / (gt.sum() + 1e-6)

            fig, axes = plt.subplots(1, 5, figsize=(25, 5))
            fig.suptitle(
                f'FSG-Net Sample {count+1}  |  Dice: {img_dice:.4f}  |  '
                f'Sensitivity: {img_sens:.4f}  |  Vessel%: {gt.mean()*100:.2f}%',
                fontsize=12, fontweight='bold'
            )
            axes[0].imshow(img);            axes[0].set_title('Original fundus image', fontsize=10);         axes[0].axis('off')
            axes[1].imshow(gt, cmap='gray');axes[1].set_title('Ground truth mask', fontsize=10);             axes[1].axis('off')
            im = axes[2].imshow(prob, cmap='hot', vmin=0, vmax=1)
            axes[2].set_title('Prediction probability (GCB output)', fontsize=10); axes[2].axis('off')
            plt.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)
            axes[3].imshow(overlay);        axes[3].set_title('Vessels overlay (red)', fontsize=10);         axes[3].axis('off')
            axes[4].imshow(err)
            tp_p = mpatches.Patch(color='green', label='TP  correct')
            fp_p = mpatches.Patch(color='red',   label='FP  false alarm')
            fn_p = mpatches.Patch(color='blue',  label='FN  missed vessel')
            axes[4].legend(handles=[tp_p, fp_p, fn_p], loc='lower right', fontsize=8)
            axes[4].set_title('Error map', fontsize=10); axes[4].axis('off')

            plt.tight_layout()
            plt.savefig(os.path.join(save_dir, f'prediction_{count+1:02d}.png'),
                        dpi=120, bbox_inches='tight')
            plt.show(); plt.close()
            count += 1

    print(f'{count} visualizations saved → {save_dir}')


visualize_predictions(model, test_loader, n_samples=8)


In [ ]:
# ============================================================
# Cell 17: Final Evaluation + Save JSON Results
# ============================================================

import os
import json
import numpy as np
import torch


def to_python(x):
    """Convert NumPy / Torch scalars to native Python types for JSON."""
    if isinstance(x, torch.Tensor):
        if x.numel() == 1:
            return x.item()
        return x.detach().cpu().tolist()
    if isinstance(x, np.generic):   # handles np.float32, np.int64, np.bool_, etc.
        return x.item()
    return x


final = {
    'model_info': {
        'architecture'       : 'FSG-Net (Full-Scale Guided Network)',
        'paper'              : 'Seo et al., BMC Medical Imaging, 2025',
        'arxiv'              : '2501.18921',
        'encoder_channels'   : [int(c) for c in CFG['encoder_channels']],
        'gcb_hidden'         : int(CFG['gcb_channels']),
        'deep_supervision'   : bool(to_python(CFG['use_deep_supervision'])),
        'img_size'           : int(CFG['img_size']),
        'n_train'            : int(len(train_dataset)),
        'n_val'              : int(len(val_dataset)),
        'n_test'             : int(len(test_dataset)),
    },
    'paper_results': {
        'DRIVE_F1'     : 84.068,
        'DRIVE_AUC'    : 98.235,
        'STARE_F1'     : 85.100,
        'CHASE_F1'     : 81.019,
        'HRF_F1'       : 81.567,
    },
    'hyperparameters': {
        'lr_encoder'     : float(to_python(CFG['lr_encoder'])),
        'lr_decoder'     : float(to_python(CFG['lr_decoder'])),
        'batch_size'     : int(CFG['batch_size']),
        'total_epochs'   : int(CFG['total_epochs']),
        'dice_weight'    : float(to_python(CFG['dice_weight'])),
        'ds_weight'      : float(to_python(CFG['ds_weight'])),
        'label_smoothing': float(to_python(CFG['label_smoothing'])),
    },
    'test_metrics': {
        'dice'        : float(to_python(test_results['dice'])),
        'iou'         : float(to_python(test_results['iou'])),
        'sensitivity' : float(to_python(test_results['sensitivity'])),
        'specificity' : float(to_python(test_results['specificity'])),
        'precision'   : float(to_python(test_results['precision'])),
        'auc_roc'     : float(to_python(test_results['auc_roc'])),
        'auc_pr'      : float(to_python(test_results['auc_pr'])),
    },
    'targets': {
        'dice_target'         : 0.82,
        'dice_met'            : bool(to_python(test_results['dice']) >= 0.82),
        'paper_dice_target'   : 0.84068,
        'paper_target_met'    : bool(to_python(test_results['dice']) >= 0.84068),
        'iou_target'          : 0.75,
        'iou_met'             : bool(to_python(test_results['iou']) >= 0.75),
        'sensitivity_target'  : 0.80,
        'sensitivity_met'     : bool(to_python(test_results['sensitivity']) >= 0.80),
        'specificity_target'  : 0.97,
        'specificity_met'     : bool(to_python(test_results['specificity']) >= 0.97),
    },
}

out_path = os.path.join(CFG['results_dir'], 'metrics', 'test_results.json')
os.makedirs(os.path.dirname(out_path), exist_ok=True)

with open(out_path, 'w') as f:
    json.dump(final, f, indent=2)

print('=' * 60)
print('  FSG-NET  —  FINAL TEST SET RESULTS')
print('=' * 60)

m = final['test_metrics']
t = final['targets']

print(f'  Dice Score   : {m["dice"]:.4f}   Phase A target >0.82 {"✅" if t["dice_met"] else "❌"}')
print(f'  Dice vs paper: {m["dice"]:.4f}   Paper DRIVE  =0.8407 {"✅" if t["paper_target_met"] else "❌"}')
print(f'  IoU          : {m["iou"]:.4f}   target >0.75         {"✅" if t["iou_met"] else "❌"}')
print(f'  Sensitivity  : {m["sensitivity"]:.4f}   target >0.80         {"✅" if t["sensitivity_met"] else "❌"}')
print(f'  Specificity  : {m["specificity"]:.4f}   target >0.97         {"✅" if t["specificity_met"] else "❌"}')
print(f'  Precision    : {m["precision"]:.4f}')
print(f'  AUC-ROC      : {m["auc_roc"]:.4f}')
print(f'  AUC-PR       : {m["auc_pr"]:.4f}')
print('=' * 60)
print(f'\n  Results saved → {out_path}')

print('\n  FSGNet_results/ — all saved files:')
for root, dirs, files in os.walk(CFG['results_dir']):
    for file in sorted(files):
        fpath = os.path.join(root, file)
        size  = os.path.getsize(fpath)
        rel   = os.path.relpath(fpath, CFG['results_dir'])
        size_str = f'{size/1024:.0f} KB' if size < 1e6 else f'{size/1e6:.1f} MB'
        print(f'    {rel:<45} {size_str}')

In [ ]:
import shutil

folder_path = "FSG_Net_results"   # <-- your folder name
output_zip = "FSG_Net_results.zip"

shutil.make_archive("FSG_Net_results", 'zip', folder_path)

print("ZIP created:", output_zip)